# 扫雷 ― 从数字提示推理安全的格子

## 本章要点

扫雷是一个靠已翻开格子上的数字来推理地雷位置的游戏。这一章，我们把这件推理工作交给 Z3。

- 用 `Bool` 变量表示未翻开的格子
- 把“周围数字＝周围地雷数”写成约束条件
- 用**反证法**确定“一定安全的格子”和“一定是地雷的格子”（`push()` / `check()` / `pop()`）

人用手指一格一格翻开的工作，就由 Z3 来代替了。

## 扫雷的规则

盘面上的数字，表示**这个格子周围 8 格中的地雷数**。数字格子是已经翻开的、安全的格子，`.` 是还没翻开的格子。

这一章要写的程序，会从给定的盘面中找出**可以断定不是地雷的格子**。

## 解题思路

对每个格子，分别试下面两种假设。

1. 假设“这个格子有地雷”，看它和约束条件是否矛盾
   - 矛盾（`unsat`）→ 不可能有地雷 → **安全格**
2. 假设“这个格子没有地雷”，看它和约束条件是否矛盾
   - 矛盾（`unsat`）→ 确定是地雷 → **地雷格**
3. 两种假设都不矛盾 → 还判断不了（运气不好，翻开就爆炸）

这就是所谓的“反证法”。加上假设后出现了矛盾，就说明假设是错的。

临时加上的假设，要用 `solver.push()`（保存状态）和 `solver.pop()`（恢复保存的状态）夹起来，不让它带到下一个格子的判断里。

## 代码

我们写三个函数。

- `get_neighbors()` … 返回某个格子周围的 8 格
- `solve_board()` … 求解盘面，返回安全格和地雷格
- `show_result()` … 用盘面的形式显示结果

In [1]:
from z3 import *

# 8 个方向（上下左右＋斜向）
DIRECTIONS = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
]


def get_neighbors(x, y, h, w):
    """返回格子 (x, y) 周围 8 格中位于盘面内的那些"""
    neighbors = []
    for dx, dy in DIRECTIONS:
        nx, ny = x + dx, y + dy
        if 0 <= nx < w and 0 <= ny < h:
            neighbors.append((nx, ny))
    return neighbors


def solve_board(board_text):
    """求解盘面，返回 (安全格, 地雷格)"""
    rows = board_text.strip().split()
    h, w = len(rows), len(rows[0])

    # 给数字格周围的未翻开格子建一个表示“有地雷”的 Bool 变量
    mine = {}
    for y, row in enumerate(rows):
        for x, v in enumerate(row):
            if v == '.':
                continue                      # 未翻开的格子本身跳过
            for nx, ny in get_neighbors(x, y, h, w):
                if rows[ny][nx] == '.':
                    mine[nx, ny] = Bool(f'mine_{nx}_{ny}')

    # 约束条件：数字格周围为 True 的变量个数 = 这个数字
    solver = Solver()
    for y, row in enumerate(rows):
        for x, v in enumerate(row):
            if v == '.':
                continue
            around = [mine[loc] for loc in get_neighbors(x, y, h, w) if loc in mine]
            if around:
                solver.add(Sum(around) == int(v))

    # 给每个格子加上假设，看是否矛盾
    safe, mines = [], []
    for (x, y), var in mine.items():
        solver.push()
        solver.add(var)                   # 假设：这里有地雷
        if solver.check() == unsat:       # 矛盾 → 不是地雷
            safe.append((x, y))
        solver.pop()

        solver.push()
        solver.add(Not(var))              # 假设：这里没有地雷
        if solver.check() == unsat:       # 矛盾 → 确定是地雷
            mines.append((x, y))
        solver.pop()

    return safe, mines


def show_result(board_text, safe, mines):
    """“#”是安全格，“*”是地雷格"""
    board = [list(row) for row in board_text.strip().split()]
    for x, y in mines:
        board[y][x] = '*'
    for x, y in safe:
        board[y][x] = '#'
    print("\n".join(" ".join(row) for row in board))

## 执行示例

![minesweeper](images/minesweeper01.png)

我们来解一下这个盘面。

In [2]:
board_str = """
...201..................
...201..................
...1013.................
...1102.................
....1011123.............
..21100012...22211122...
..2000012.33212.3111....
..421002.310002.3.122...
....3224.31000112111....
........4.20000001121...
.......43.20000012.11...
.......3122211001.212...
.......202.3.10011213...
......2103.42210001.213.
......1112.43.111233202.
......2.222..311.3..212.
.............2124.4.....
............111.3.......
........................
........................
"""

safe, mines = solve_board(board_str)
print(f"安全格: {len(safe)} 个")
print(f"地雷格: {len(mines)} 个")
show_result(board_str, safe, mines)

安全格: 43 个
地雷格: 54 个
. . * 2 0 1 # . . . . . . . . . . . . . . . . .
. . * 2 0 1 * # . . . . . . . . . . . . . . . .
. . # 1 0 1 3 * . . . . . . . . . . . . . . . .
. . # 1 1 0 2 * # # * # . . . . . . . . . . . .
. # # * 1 0 1 1 1 2 3 # # # * # # # * # # . . .
. . 2 1 1 0 0 0 1 2 * * * 2 2 2 1 1 1 2 2 . . .
. . 2 0 0 0 0 1 2 * 3 3 2 1 2 * 3 1 1 1 * # . .
. * 4 2 1 0 0 2 * 3 1 0 0 0 2 * 3 * 1 2 2 # . .
. . * * 3 2 2 4 * 3 1 0 0 0 1 1 2 1 1 1 * # . .
. . . * * # * * 4 * 2 0 0 0 0 0 0 1 1 2 1 # . .
. . . . . . * 4 3 * 2 0 0 0 0 0 1 2 * 1 1 # . .
. . . . . . * 3 1 2 2 2 1 1 0 0 1 * 2 1 2 * . .
. . . . . * * 2 0 2 * 3 * 1 0 0 1 1 2 1 3 * # .
. . . . . # 2 1 0 3 * 4 2 2 1 0 0 0 1 * 2 1 3 .
. . . . . # 1 1 1 2 * 4 3 * 1 1 1 2 3 3 2 0 2 .
. . . . . # 2 * 2 2 2 * * 3 1 1 * 3 * * 2 1 2 .
. . . . . . . # * # # # * 2 1 2 4 * 4 # . . # .
. . . . . . . . . . . # 1 1 1 * 3 * # # . . . .
. . . . . . . . . . . # # # # # # # . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . .


未翻开的格子有 111 个，其中 **43 个安全**、**54 个是地雷**，已经确定。剩下的 14 个，不管假设哪一种都不矛盾，所以判断不了。人玩的时候，这 14 个也是到最后都不知道（只能碰运气）的格子。

盘面上的 `#` 是安全格，`*` 是地雷格。

## 代码讲解

### 1. 创建变量

```python
mine = {}
for y, row in enumerate(rows):
    for x, v in enumerate(row):
        if v == '.':
            continue
        for nx, ny in get_neighbors(x, y, h, w):
            if rows[ny][nx] == '.':
                mine[ny, nx] = Bool(f'mine_{ny}_{nx}')
```

`mine` 是 `(x, y) → Bool 变量` 的字典。变量的含义是“这个格子有没有地雷”，`True` 表示有地雷，`False` 表示安全。

只给数字格周围的未翻开格子创建变量。没和数字挨着的格子，一点提示都没有，也就无从判断。

### 2. 添加约束条件

```python
around = [mine[loc] for loc in get_neighbors(x, y, h, w) if loc in mine]
if around:
    solver.add(Sum(around) == int(v))
```

对数字格 `v`，添加“周围未翻开格子中的地雷数”等于 `v` 的约束条件。

`Sum(around)` 是 `Bool` 变量的和。Z3 会把 `True` 当作 1、`False` 当作 0 来相加，所以直接就是地雷的个数。

`around` 为空的情况，是数字周围全都已经翻开的时候。那时没有要加的约束，所以用 `if around:` 跳过。

### 3. 用反证法判断

```python
solver.push()
solver.add(var)                   # 假设：这里有地雷
if solver.check() == unsat:       # 矛盾 → 不是地雷
    safe.append((x, y))
solver.pop()
```

| 调用 | 作用 |
|---|---|
| `solver.push()` | 保存当前的约束，以便加上临时约束 |
| `solver.add(...)` | 添加假设 |
| `solver.check()` | 在加上假设的状态下检查有没有解（`unsat` 就是矛盾） |
| `solver.pop()` | 恢复到保存的状态（丢掉假设） |

忘了 `pop()`，前一个格子的假设就会留到下一个格子的判断里，答案就乱了。`push()` 和 `pop()` 一定要成对使用。

## 小结

- 把未翻开的格子表示成 `Bool` 变量，“周围的地雷数”就能写成 `Sum(...) == 数字`
- 假设“有地雷”时 `unsat` 就是安全，假设“没有地雷”时 `unsat` 就确定是地雷（反证法）
- 临时假设要用 `push()` 和 `pop()` 夹起来，不要带到下一次判断
- 两种假设都不矛盾的格子判断不了。扫雷里“运气不好就会爆炸”，原因就在这里